# 06 · Decision Framework

When do you reach for which tool? The whole series in one guide.

| Use... | When | Avoid when |
|---|---|---|
| **plain SQL** | shallow 1–2 hop relationships you already query relationally | multi-hop / variable-length / cycles — it balloons |
| **Apache AGE** | your data is already in Postgres and you need multi-hop traversal, path patterns, or cycle detection *next to it* | you need built-in graph *algorithms* (PageRank, communities) |
| **NetworkX / igraph** | the graph fits in memory and you need graph algorithms or one-off analysis | the graph is huge, or you want it to live next to your tables |
| **Neo4j + GDS** | very large graphs, heavy graph OLTP, or a mature built-in algorithms library | a modest graph that already lives in Postgres |

And the combined pattern that is often best: **keep the graph in AGE next to your tables,
and pull subgraphs into Python for the heavy algorithms.**

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np

from src.age.connect import age_available, get_connection
from src.config import load_settings
from src.datasets.graphs import KNOWS, PAYS, all_graphs
from src.outputs import figure_path, save_json
from src.queries.rings import find_rings_age, find_rings_networkx
from src.queries.traversal import reachable_age, reachable_networkx, variable_length_cypher

settings = load_settings()
AGE = age_available(settings)
social = all_graphs(settings)["social_txn"]
GRAPH = settings.graphs.social
print("AGE:", AGE)

AGE: True


## The framework, as data

Saved to JSON so the article and the app can render it.

In [2]:
framework = [
    {"tool": "plain SQL",
     "use_when": "shallow 1-2 hop relationships you already query relationally",
     "avoid_when": "multi-hop / variable-length / cycle detection"},
    {"tool": "Apache AGE",
     "use_when": "multi-hop traversal, path patterns, cycle detection on data already in Postgres",
     "avoid_when": "you need built-in graph algorithms (PageRank, communities, centrality)"},
    {"tool": "NetworkX / igraph",
     "use_when": "graph fits in memory and you need graph algorithms or one-off analysis",
     "avoid_when": "the graph is huge, or it should live next to your tables"},
    {"tool": "Neo4j + GDS",
     "use_when": "very large graphs, heavy graph OLTP, mature built-in algorithms library",
     "avoid_when": "a modest graph that already lives in Postgres"},
    {"tool": "AGE + Python (combined)",
     "use_when": "keep the graph in AGE next to your tables; pull subgraphs into Python for heavy algorithms",
     "avoid_when": "rarely — this is the default sweet spot for Postgres shops"},
]
print("saved ->", save_json("06_decision_framework.json", framework))

saved -> /workspaces/apache_age_101/outputs/data/06_decision_framework.json


## Figure: a capability matrix

Editorial scores — 0 poor, 1 workable, 2 strong — across the questions this series asked.

In [3]:
rows = ["plain SQL", "Apache AGE", "NetworkX / igraph", "Neo4j + GDS"]
cols = ["shallow\n1-2 hop", "multi-hop\ntraversal", "cycle / ring\ndetection",
        "in-memory\nalgorithms", "very large\n/ OLTP", "next to\nPostgres data"]
scores = np.array([
    [2, 0, 0, 0, 1, 2],   # plain SQL
    [2, 2, 2, 0, 1, 2],   # Apache AGE
    [1, 2, 2, 2, 0, 0],   # NetworkX / igraph
    [1, 2, 2, 2, 2, 0],   # Neo4j + GDS
])

fig, ax = plt.subplots(figsize=(9, 4))
im = ax.imshow(scores, cmap="YlGn", vmin=0, vmax=2, aspect="auto")
ax.set_xticks(range(len(cols)), cols, fontsize=8)
ax.set_yticks(range(len(rows)), rows)
labels = {0: "", 1: "ok", 2: "✓"}
for i in range(len(rows)):
    for j in range(len(cols)):
        ax.text(j, i, labels[scores[i, j]], ha="center", va="center", fontsize=11)
ax.set_title("Which tool for which graph question")
fig.colorbar(im, ax=ax, ticks=[0, 1, 2], shrink=0.7).set_label("fit (0-2)")
fig.tight_layout()
fig.savefig(figure_path("06_decision_matrix.png"), dpi=120)
plt.close(fig)
save_json("06_decision_matrix.json",
          {"rows": rows, "cols": [c.replace("\n", " ") for c in cols], "scores": scores.tolist()})
print("saved matrix")

saved matrix


## The toolkit you leave with

Everything in this series is one small library. A quick end-to-end recap — the one-line
Cypher traversal and the ring pattern — run through the same helpers the notebooks used.

In [4]:
if AGE:
    conn = get_connection(settings)
    reachable = reachable_age(conn, GRAPH, 0, 1, 3, KNOWS) - {0}
    rings = find_rings_age(conn, GRAPH, 3, 5, PAYS)
    conn.close()
    source = "AGE (Cypher, in Postgres)"
else:
    reachable = reachable_networkx(social, 0, 1, 3, KNOWS)
    rings = find_rings_networkx(social, 3, 5, PAYS)
    source = "NetworkX baseline"

recap = {
    "source": source,
    "one_line_cypher": variable_length_cypher(0, 1, 3, KNOWS),
    "reachable_within_3_hops_of_account_0": len(reachable),
    "fraud_rings_found": len(rings),
    "planted_rings": len(social.rings),
    "helpers": [
        "src.age.connect.get_connection  (LOAD age + search_path)",
        "src.age.cypher.run_cypher       (the one Cypher adapter)",
        "src.queries.{traversal,paths,rings,patterns}  (three implementations each)",
        "src.age.to_networkx.subgraph_to_networkx  (AGE -> NetworkX bridge)",
        "src.evaluation.comparison.compare  (the arena harness)",
    ],
}
print(f"{source}: {len(reachable)} accounts within 3 hops of 0; {len(rings)} fraud rings")
print("saved ->", save_json("06_recap.json", recap))

AGE (Cypher, in Postgres): 183 accounts within 3 hops of 0; 6 fraud rings
saved -> /workspaces/apache_age_101/outputs/data/06_recap.json


**Takeaway.** The graph database everyone assumes they must install is often hiding
inside the Postgres they already run. Use AGE when your data lives in Postgres and the
questions are traversal-shaped; pull subgraphs into Python when you need real graph
algorithms; reach for a dedicated graph platform only when scale or built-in analytics
truly demand it. Right tool, right question — now you can tell which is which.